# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* **PEFT technique**: LoRA (Low-Rank Adaptation)
* **Model**: GPT-2 (`gpt2`) 
* **Evaluation approach**: HuggingFace `Trainer` with `evaluate` accuracy metric, comparing before and after fine-tuning
* **Fine-tuning dataset**: GLUE SST-2 

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [ ]:

import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    GPT2Tokenizer,
    GPT2ForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)



In [ ]:
dataset = load_dataset("glue", "sst2")
print(dataset)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["sentence", "idx"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

print("\nTokenized dataset:", tokenized)
print("Example features:", tokenized["train"].features)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

Tokenized dataset: DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1821
    })
})
Example features: {'labels': ClassLabel(names=['negative', 'positive']), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


In [ ]:
model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=2)
model.config.pad_token_id = tokenizer.eos_token_id

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(model)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5734.16it/s]
GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 124,441,344
GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (score): Linear(in_features=768, out_features=2, bias=False)
)


In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

eval_args = TrainingArguments(
    output_dir="/tmp/foundation_eval",
    per_device_eval_batch_size=32,
    report_to="none",
)

trainer_base = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback
trainer_base.remove_callback(NotebookProgressCallback)


results_before = trainer_base.evaluate()
print("Foundation model (before fine-tuning):", results_before)

/Users/marinalee/Desktop/udacity_genai/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Foundation model (before fine-tuning): {'eval_loss': 0.7722347974777222, 'eval_model_preparation_time': 0.0008, 'eval_accuracy': 0.4919724770642202, 'eval_runtime': 3.1675, 'eval_samples_per_second': 275.299, 'eval_steps_per_second': 8.84, 'epoch': 0}


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,               
    lora_alpha=32,   
    lora_dropout=0.1,
    target_modules=["c_attn"], 
    bias="none",
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 296,448 || all params: 124,737,792 || trainable%: 0.2377


/Users/marinalee/Desktop/udacity_genai/.venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
train_args = TrainingArguments(
    output_dir="/tmp/peft_gpt2_sst2",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

train_subset = tokenized["train"].select(range(3000))

trainer_peft = Trainer(
    model=peft_model,
    args=train_args,
    train_dataset=train_subset,
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer_peft.train()

/Users/marinalee/Desktop/udacity_genai/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.589285,0.737385


TrainOutput(global_step=188, training_loss=0.6637722177708403, metrics={'train_runtime': 24.9668, 'train_samples_per_second': 120.159, 'train_steps_per_second': 7.53, 'total_flos': 52756956094464.0, 'train_loss': 0.6637722177708403, 'epoch': 1.0})

###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [ ]:
peft_model.save_pretrained("peft_gpt2_sst2")
tokenizer.save_pretrained("peft_gpt2_sst2")
print("PEFT adapter weights saved to peft_gpt2_sst2")

PEFT adapter weights saved to peft_gpt2_sst2


## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [ ]:
from peft import PeftModel

base_model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=2)
base_model.config.pad_token_id = tokenizer.eos_token_id

loaded_peft_model = PeftModel.from_pretrained(base_model, "peft_gpt2_sst2")
loaded_peft_model.eval()
print("PEFT model loaded from peft_gpt2_sst2")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6622.61it/s]
GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PEFT model loaded from peft_gpt2_sst2


In [ ]:
eval_args_peft = TrainingArguments(
    output_dir="/tmp/peft_eval",
    per_device_eval_batch_size=32,
    report_to="none",
)

trainer_eval = Trainer(
    model=loaded_peft_model,
    args=eval_args_peft,
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback
trainer_eval.remove_callback(NotebookProgressCallback)

results_after = trainer_eval.evaluate()
print("Fine-tuned model (after LoRA):", results_after)

/Users/marinalee/Desktop/udacity_genai/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Fine-tuned model (after LoRA): {'eval_loss': 0.5892848968505859, 'eval_model_preparation_time': 0.0015, 'eval_accuracy': 0.7373853211009175, 'eval_runtime': 3.3334, 'eval_samples_per_second': 261.598, 'eval_steps_per_second': 8.4, 'epoch': 0}


In [ ]:
print("=" * 45)
print(f"  Foundation model accuracy: {results_before['eval_accuracy']:.4f}")
print(f"  Fine-tuned model accuracy: {results_after['eval_accuracy']:.4f}")
improvement = results_after['eval_accuracy'] - results_before['eval_accuracy']
print(f"  Improvement:              {improvement:+.4f}")
print("=" * 45)

  Foundation model accuracy: 0.4920
  Fine-tuned model accuracy: 0.7374
  Improvement:              +0.2454
